# 00 — Mac Smoke Test (SMPL-X loading + forward + rendering)

**Purpose**: confirm that SMPL-X library + matplotlib rendering work on the local Mac
(M-series, Python 3.11) before moving to Colab for the full pipeline.

Pre-conditions:
- `bash demo/scripts/setup_smpl_paths.sh` has been run
- venv with `torch / smplx / scipy / trimesh / matplotlib / imageio` installed

Expected runtime: < 10 seconds end-to-end.

In [ ]:
import sys, os, time
from pathlib import Path
# Make the repo root importable (so `from demo.src ...` works)
REPO = Path.cwd().parent.parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO))

import torch
print(f"PyTorch {torch.__version__}, MPS available: {torch.backends.mps.is_available()}")

## Test 1 — SMPL-X T-pose forward

In [ ]:
from demo.src.smpl_forward import smpl_forward_eq1, export_tpose_obj

t0 = time.time()
verts, joints, model = smpl_forward_eq1(model_type='smplx')
print(f"Forward: {(time.time()-t0)*1000:.1f}ms  vertices: {tuple(verts.shape)}  joints: {tuple(joints.shape)}")

obj = export_tpose_obj('../results/tpose_smplx.obj', model_type='smplx')
print(f"Saved: {obj}  ({os.path.getsize(obj)//1024} KB)")

## Test 2 — Render 4 different SMPL-X poses

In [ ]:
import matplotlib.pyplot as plt
from demo.src.visualize import render_mesh_matplotlib

poses = {'tpose': torch.zeros(1,63)}
p = torch.zeros(1,63); p[0, 17*3+1] = 1.4;  poses['arm_raised'] = p
p = torch.zeros(1,63); p[0, 5*3+0]  = -0.4; poses['leaning']    = p
p = torch.zeros(1,63); p[0, 0*3+1]  = 0.8;  poses['twist']      = p

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for ax, (name, pose) in zip(axes, poses.items()):
    out = model(betas=torch.zeros(1,10), body_pose=pose, global_orient=torch.zeros(1,3))
    img = render_mesh_matplotlib(out.vertices.squeeze().detach().numpy(), model.faces, azim=30, elev=10, figsize=(4,4))
    ax.imshow(img); ax.set_title(name); ax.axis('off')
plt.show()

## Test 3 — MPS speedup benchmark

In [ ]:
import smplx
from demo.src import MODEL_ROOT

if torch.backends.mps.is_available():
    smpl_mps = smplx.create(str(MODEL_ROOT), model_type='smplx', gender='neutral',
                             num_betas=10, num_expression_coeffs=10, ext='npz').to('mps')
    B = 32
    betas = torch.zeros(B, 10).to('mps')
    bp    = torch.zeros(B, 63).to('mps')
    go    = torch.zeros(B, 3).to('mps')
    for _ in range(5):  # warmup
        _ = smpl_mps(betas=betas, body_pose=bp, global_orient=go); torch.mps.synchronize()
    t0 = time.time()
    for _ in range(20):
        _ = smpl_mps(betas=betas, body_pose=bp, global_orient=go)
    torch.mps.synchronize()
    ms_per_iter = (time.time() - t0) / 20 * 1000
    print(f"MPS @ B={B}: {ms_per_iter:.2f} ms/iter  →  {ms_per_iter/B:.3f} ms/sample")
else:
    print("MPS not available; running CPU only")

## ✅ If all three tests pass

You can now safely move to `01_main_pipeline.ipynb` (which runs on Colab and uses
the full HRNet + 4D-Humans stack).